# Task 4: Model Evaluation & Comparison

## Objective
This notebook covers:
1. Comprehensive evaluation with multiple metrics
2. Confusion matrices for all models
3. ROC curves and AUC scores
4. Precision-Recall curves
5. Error analysis
6. Threshold optimization
7. Model comparison and selection

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
    average_precision_score
)
import joblib

# Set display and plot options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

## 4.1 Load Models and Data

Note: If models haven't been trained yet (Task 3), this notebook provides the evaluation framework that will be used once models are available.

In [ ]:
import os

# Check if models exist
models_dir = '../models'
model_files = {
    'Logistic Regression': 'logistic_regression.pkl',
    'Random Forest': 'random_forest.pkl',
    'XGBoost': 'xgboost.pkl',
    'LightGBM': 'lightgbm.pkl'
}

available_models = {}
for name, filename in model_files.items():
    filepath = os.path.join(models_dir, filename)
    if os.path.exists(filepath):
        available_models[name] = joblib.load(filepath)
        print(f"✓ Loaded {name}")
    else:
        print(f"⚠ {name} not found at {filepath}")

if len(available_models) == 0:
    print("\n⚠ No models found. Please run Task 3 (Model Development) first.")
    print("This notebook will demonstrate the evaluation framework with placeholder metrics.")
else:
    print(f"\n✓ Successfully loaded {len(available_models)} models")

In [ ]:
# Load test data
try:
    X_test = np.load('../data/processed/X_test_scaled.npy')
    y_test = np.load('../data/processed/y_test.npy')
    print(f"✓ Loaded test data: X_test {X_test.shape}, y_test {y_test.shape}")
except:
    print("⚠ Test data not found. Using placeholder data for demonstration.")
    # Create placeholder data
    from sklearn.datasets import make_classification
    X_test, y_test = make_classification(n_samples=1000, n_features=50, n_classes=2, 
                                         weights=[0.88, 0.12], random_state=42)
    print(f"Generated placeholder data: X_test {X_test.shape}, y_test {y_test.shape}")

## 4.2 Comprehensive Metrics Evaluation

### Classification Metrics:
1. **Accuracy**: Overall correctness
2. **Precision**: Of predicted positives, how many are actually positive
3. **Recall (Sensitivity)**: Of actual positives, how many did we find
4. **F1-Score**: Harmonic mean of precision and recall
5. **ROC-AUC**: Area under ROC curve
6. **Average Precision**: Area under precision-recall curve

In [ ]:
# Evaluation function
def evaluate_model(model, X_test, y_test, model_name):
    """
    Comprehensive model evaluation.
    """
    # Predictions
    y_pred = model.predict(X_test)
    
    # Probabilities
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
    except:
        y_proba = y_pred
    
    # Calculate metrics
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'Avg Precision': average_precision_score(y_test, y_proba)
    }
    
    return metrics, y_pred, y_proba

# Evaluate all models
all_metrics = []
all_predictions = {}
all_probabilities = {}

if len(available_models) > 0:
    for name, model in available_models.items():
        metrics, y_pred, y_proba = evaluate_model(model, X_test, y_test, name)
        all_metrics.append(metrics)
        all_predictions[name] = y_pred
        all_probabilities[name] = y_proba
        print(f"✓ Evaluated {name}")
    
    # Create results dataframe
    results_df = pd.DataFrame(all_metrics)
    results_df = results_df.sort_values('F1-Score', ascending=False)
    
    print("\n" + "="*80)
    print("MODEL EVALUATION RESULTS")
    print("="*80)
    display(results_df.round(4))
    
    # Save results
    os.makedirs('../reports', exist_ok=True)
    results_df.to_csv('../reports/detailed_evaluation.csv', index=False)
    print("\n✓ Results saved to reports/detailed_evaluation.csv")
else:
    print("\nDemonstration: Sample evaluation metrics table")
    # Create sample metrics for demonstration
    results_df = pd.DataFrame([
        {'Model': 'XGBoost', 'Accuracy': 0.912, 'Precision': 0.654, 'Recall': 0.632, 'F1-Score': 0.643, 'ROC-AUC': 0.875, 'Avg Precision': 0.598},
        {'Model': 'LightGBM', 'Accuracy': 0.908, 'Precision': 0.645, 'Recall': 0.628, 'F1-Score': 0.636, 'ROC-AUC': 0.871, 'Avg Precision': 0.591},
        {'Model': 'Random Forest', 'Accuracy': 0.905, 'Precision': 0.638, 'Recall': 0.615, 'F1-Score': 0.626, 'ROC-AUC': 0.865, 'Avg Precision': 0.585},
        {'Model': 'Neural Network', 'Accuracy': 0.902, 'Precision': 0.631, 'Recall': 0.608, 'F1-Score': 0.619, 'ROC-AUC': 0.858, 'Avg Precision': 0.578},
        {'Model': 'Logistic Regression', 'Accuracy': 0.895, 'Precision': 0.612, 'Recall': 0.595, 'F1-Score': 0.603, 'ROC-AUC': 0.842, 'Avg Precision': 0.562}
    ])
    display(results_df)

## 4.3 Confusion Matrices

In [ ]:
# Plot confusion matrices for all models
if len(available_models) > 0:
    n_models = len(available_models)
    n_cols = min(3, n_models)
    n_rows = int(np.ceil(n_models / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    if n_models == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, (name, y_pred) in enumerate(all_predictions.items()):
        cm = confusion_matrix(y_test, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                   xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
        axes[idx].set_title(f'{name}\nConfusion Matrix', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Predicted', fontsize=10)
        axes[idx].set_ylabel('Actual', fontsize=10)
    
    # Hide extra subplots
    for idx in range(n_models, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('../reports/figures/04_confusion_matrices.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Confusion matrices will be generated once models are trained.")

## 4.4 ROC Curves

In [ ]:
# Plot ROC curves for all models
if len(available_models) > 0:
    plt.figure(figsize=(10, 8))
    
    for name, y_proba in all_probabilities.items():
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc_score = roc_auc_score(y_test, y_proba)
        plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {auc_score:.3f})')
    
    # Diagonal line
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
    
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../reports/figures/04_roc_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("ROC curves will be generated once models are trained.")
    # Show example
    plt.figure(figsize=(10, 8))
    # Generate example curves
    fpr = np.linspace(0, 1, 100)
    models = ['XGBoost', 'LightGBM', 'Random Forest', 'Neural Network', 'Logistic Regression']
    aucs = [0.875, 0.871, 0.865, 0.858, 0.842]
    for model, auc in zip(models, aucs):
        # Approximate TPR curve
        tpr = np.power(fpr, 0.5) * (auc / 0.85)
        plt.plot(fpr, tpr, linewidth=2, label=f'{model} (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curves - Model Comparison (Example)', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../reports/figures/04_roc_curves_example.png', dpi=300, bbox_inches='tight')
    plt.show()

## 4.5 Precision-Recall Curves

In [ ]:
# Plot Precision-Recall curves
if len(available_models) > 0:
    plt.figure(figsize=(10, 8))
    
    for name, y_proba in all_probabilities.items():
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        ap_score = average_precision_score(y_test, y_proba)
        plt.plot(recall, precision, linewidth=2, label=f'{name} (AP = {ap_score:.3f})')
    
    # Baseline
    baseline = y_test.sum() / len(y_test)
    plt.axhline(y=baseline, color='k', linestyle='--', linewidth=1, 
                label=f'Baseline (AP = {baseline:.3f})')
    
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title('Precision-Recall Curves - Model Comparison', fontsize=14, fontweight='bold')
    plt.legend(loc='lower left', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../reports/figures/04_precision_recall_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Precision-Recall curves will be generated once models are trained.")

## 4.6 Error Analysis

Analyze which types of samples are misclassified most often.

In [ ]:
if len(available_models) > 0:
    # For the best model, analyze errors
    best_model_name = results_df.iloc[0]['Model']
    best_model = available_models[best_model_name]
    best_predictions = all_predictions[best_model_name]
    
    print("="*80)
    print(f"ERROR ANALYSIS FOR BEST MODEL: {best_model_name}")
    print("="*80)
    
    # Calculate error types
    cm = confusion_matrix(y_test, best_predictions)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"\nConfusion Matrix Breakdown:")
    print(f"  True Negatives (TN): {tn:,} - Correctly predicted 'No'")
    print(f"  False Positives (FP): {fp:,} - Incorrectly predicted 'Yes' (Type I Error)")
    print(f"  False Negatives (FN): {fn:,} - Incorrectly predicted 'No' (Type II Error)")
    print(f"  True Positives (TP): {tp:,} - Correctly predicted 'Yes'")
    
    print(f"\nError Rates:")
    print(f"  False Positive Rate: {fp / (fp + tn):.4f} (of all negatives, % incorrectly classified)")
    print(f"  False Negative Rate: {fn / (fn + tp):.4f} (of all positives, % incorrectly classified)")
    
    # Visualize error distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Error counts
    error_counts = pd.Series({
        'Correct\nPredictions': tn + tp,
        'False\nPositives': fp,
        'False\nNegatives': fn
    })
    colors = ['green', 'orange', 'red']
    error_counts.plot(kind='bar', ax=axes[0], color=colors)
    axes[0].set_title('Prediction Distribution', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_xlabel('')
    for i, v in enumerate(error_counts):
        axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')
    
    # Error rates
    error_rates = pd.Series({
        'False\nPositive\nRate': fp / (fp + tn),
        'False\nNegative\nRate': fn / (fn + tp)
    })
    error_rates.plot(kind='bar', ax=axes[1], color=['orange', 'red'])
    axes[1].set_title('Error Rates', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Rate', fontsize=12)
    axes[1].set_xlabel('')
    axes[1].set_ylim([0, max(error_rates) * 1.2])
    for i, v in enumerate(error_rates):
        axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../reports/figures/04_error_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Business interpretation
    print("\n" + "="*80)
    print("BUSINESS INTERPRETATION")
    print("="*80)
    print(f"\nFalse Positives ({fp}): Customers predicted to subscribe but didn't")
    print("  - Cost: Wasted marketing effort on uninterested customers")
    print("  - Impact: Resource allocation inefficiency")
    
    print(f"\nFalse Negatives ({fn}): Customers who would subscribe but not contacted")
    print("  - Cost: Lost revenue opportunities")
    print("  - Impact: Missed potential customers")
    
    print("\nRecommendation:")
    if fn > fp:
        print("  - Focus on reducing False Negatives (increase Recall)")
        print("  - Consider lowering classification threshold")
        print("  - Better to contact more customers than miss opportunities")
    else:
        print("  - Current balance favors Precision over Recall")
        print("  - Efficient targeting of high-probability customers")
        print("  - May want to increase threshold for even higher precision")
else:
    print("Error analysis will be performed once models are trained.")

## 4.7 Threshold Optimization

Find the optimal classification threshold to maximize F1-score or balance precision/recall.

In [ ]:
if len(available_models) > 0:
    best_model_name = results_df.iloc[0]['Model']
    best_proba = all_probabilities[best_model_name]
    
    print("="*80)
    print(f"THRESHOLD OPTIMIZATION FOR: {best_model_name}")
    print("="*80)
    
    # Test different thresholds
    thresholds = np.arange(0.1, 0.9, 0.05)
    metrics_by_threshold = []
    
    for threshold in thresholds:
        y_pred_thresh = (best_proba >= threshold).astype(int)
        precision = precision_score(y_test, y_pred_thresh, zero_division=0)
        recall = recall_score(y_test, y_pred_thresh, zero_division=0)
        f1 = f1_score(y_test, y_pred_thresh, zero_division=0)
        
        metrics_by_threshold.append({
            'Threshold': threshold,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        })
    
    threshold_df = pd.DataFrame(metrics_by_threshold)
    
    # Find optimal threshold for F1
    optimal_idx = threshold_df['F1-Score'].idxmax()
    optimal_threshold = threshold_df.loc[optimal_idx, 'Threshold']
    optimal_f1 = threshold_df.loc[optimal_idx, 'F1-Score']
    
    print(f"\nOptimal Threshold for F1-Score: {optimal_threshold:.2f}")
    print(f"Optimal F1-Score: {optimal_f1:.4f}")
    print(f"\nMetrics at Optimal Threshold:")
    print(f"  Precision: {threshold_df.loc[optimal_idx, 'Precision']:.4f}")
    print(f"  Recall: {threshold_df.loc[optimal_idx, 'Recall']:.4f}")
    
    # Visualize threshold impact
    plt.figure(figsize=(12, 6))
    plt.plot(threshold_df['Threshold'], threshold_df['Precision'], 
             linewidth=2, label='Precision', marker='o', markersize=4)
    plt.plot(threshold_df['Threshold'], threshold_df['Recall'], 
             linewidth=2, label='Recall', marker='s', markersize=4)
    plt.plot(threshold_df['Threshold'], threshold_df['F1-Score'], 
             linewidth=2, label='F1-Score', marker='^', markersize=4)
    
    # Mark optimal point
    plt.axvline(x=optimal_threshold, color='red', linestyle='--', 
                linewidth=1, label=f'Optimal Threshold = {optimal_threshold:.2f}')
    
    plt.xlabel('Classification Threshold', fontsize=12)
    plt.ylabel('Score', fontsize=12)
    plt.title(f'Threshold Optimization - {best_model_name}', fontsize=14, fontweight='bold')
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../reports/figures/04_threshold_optimization.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Save threshold results
    threshold_df.to_csv('../reports/threshold_optimization.csv', index=False)
    print("\n✓ Threshold optimization results saved to reports/threshold_optimization.csv")
else:
    print("Threshold optimization will be performed once models are trained.")

## 4.8 Final Model Selection and Justification

In [ ]:
if len(available_models) > 0:
    print("="*80)
    print("FINAL MODEL SELECTION")
    print("="*80)
    
    best_model_name = results_df.iloc[0]['Model']
    best_metrics = results_df.iloc[0]
    
    print(f"\n🏆 Best Model: {best_model_name}")
    print(f"\nPerformance Metrics:")
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Avg Precision']:
        print(f"  {metric}: {best_metrics[metric]:.4f}")
    
    print("\n" + "="*80)
    print("JUSTIFICATION FOR USE CASE")
    print("="*80)
    
    print("\nWhy these metrics are optimal for bank marketing:")
    print("\n1. F1-Score Balance:")
    print("   - Balances precision and recall")
    print("   - Important when both false positives and false negatives have costs")
    print("   - False Positive: Wasted contact on uninterested customer")
    print("   - False Negative: Missed opportunity with interested customer")
    
    print("\n2. High Precision:")
    print("   - When we predict 'Yes', we're usually right")
    print("   - Efficient use of marketing resources")
    print("   - Better customer experience (fewer unwanted calls)")
    
    print("\n3. Good Recall:")
    print("   - We capture a good portion of potential customers")
    print("   - Not missing too many revenue opportunities")
    print("   - Critical for business growth")
    
    print("\n4. ROC-AUC Score:")
    print("   - Shows model's ability to discriminate between classes")
    print("   - Higher AUC = better ranking of customers by subscription probability")
    print("   - Allows flexible threshold adjustment based on business needs")
    
    print("\n" + "="*80)
    print("DEPLOYMENT RECOMMENDATION")
    print("="*80)
    print(f"\nRecommended Model: {best_model_name}")
    print(f"Recommended Threshold: {optimal_threshold if 'optimal_threshold' in locals() else 0.5:.2f}")
    print("\nExpected Benefits:")
    print("  • Improved targeting efficiency")
    print("  • Reduced marketing costs")
    print("  • Better customer satisfaction")
    print("  • Increased conversion rate")
    print("  • Higher ROI on marketing campaigns")
else:
    print("\n" + "="*80)
    print("MODEL SELECTION FRAMEWORK READY")
    print("="*80)
    print("\nOnce models are trained (Task 3), this notebook will:")
    print("  1. Compare all models across multiple metrics")
    print("  2. Generate comprehensive visualizations")
    print("  3. Perform error analysis")
    print("  4. Optimize classification threshold")
    print("  5. Recommend best model for deployment")

## 4.9 Summary

### Evaluation Framework Completed
✓ Multiple evaluation metrics (Accuracy, Precision, Recall, F1, ROC-AUC, AP)
✓ Confusion matrices for all models
✓ ROC curves for model comparison
✓ Precision-Recall curves
✓ Error analysis and business interpretation
✓ Threshold optimization
✓ Model selection with justification

### Key Insights
1. **Boosting models** (XGBoost, LightGBM) typically perform best on this dataset
2. **Class imbalance** handling with SMOTE significantly improves recall
3. **Threshold tuning** can optimize for business objectives
4. **F1-Score** is the most appropriate metric for this use case
5. **ROC-AUC** shows model's discrimination ability

### Next Steps (Task 5)
- Model interpretability with SHAP
- Feature importance analysis
- Partial dependence plots
- Business insights and recommendations